# 04 — UC2 Results & Comparative Analysis

Load all experiment results and produce the key analysis plots:
1. **Alpha sensitivity**: MAE vs α per algorithm
2. **Communication–Accuracy Pareto frontier**
3. **Per-deployment equity**: box plots of per-user MAE
4. **Training convergence comparison**
5. **Summary table** for the TFG

In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import Counter


sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

uc2.setup_plot_style()

## 1. Load All Results

In [ ]:
# Load standard algorithms
results = {}

for alg in ["Centralized", "FedAvg", "FedGen"]:
    for alpha in uc2.ALPHA_SWEEP:
        r = uc2.load_result(alg, alpha)
        if r is not None:
            results[(alg, alpha)] = r
            print(f"  ✓ {alg} α={alpha}: {r['n_rounds']} rounds")

# Load FedGen partial results
for alpha in uc2.ALPHA_SWEEP:
    path = os.path.join(
        uc2.RESULTS, "fedgen_partial", f"alpha_{alpha}",
        "lstm", "rep_0", "full_results.pkl"
    )
    if os.path.exists(path):
        import pickle
        with open(path, "rb") as f:
            r = pickle.load(f)
        r["algorithm"] = "FedGen_partial"
        results[("FedGen_partial", alpha)] = r
        print(f"  ✓ FedGen_partial α={alpha}: {r['n_rounds']} rounds")

print(f"\nTotal experiments loaded: {len(results)}")

In [ ]:
# Build summary table
rows = uc2.extract_final_metrics(results)
df = pd.DataFrame(rows)
print(df[["algorithm", "alpha", "best_round", "n_rounds", 
          "unscaled_mae", "unscaled_mape", "per_user_mae_std", 
          "per_user_mae_cv"]].to_string(index=False))

## 2. Alpha Sensitivity Plot

**Key plot for the TFG**: How does each algorithm degrade as 
data heterogeneity increases (lower α)?

In [ ]:
fig, ax = uc2.plot_alpha_sensitivity(rows, metric="unscaled_mae",
                                      ylabel="Unscaled MAE (MB)")
plt.savefig(os.path.join(uc2.RESULTS, "alpha_sensitivity.pdf"),
            bbox_inches="tight", dpi=150)
plt.show()

## 3. Communication–Accuracy Pareto Frontier

**Key plot for the TFG**: Each point = (comm cost, MAE) for one 
(algorithm, α) combo. FedGen partial should dominate the lower-left.

In [ ]:
costs = uc2.compute_comm_costs(results, n_users_per_round=20)

fig, ax = uc2.plot_pareto_frontier(rows, costs, metric="unscaled_mae",
                                    ylabel="Unscaled MAE (MB)")
plt.savefig(os.path.join(uc2.RESULTS, "pareto_frontier.pdf"),
            bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
# Print Pareto table
print(f"{'Algorithm':<18} {'α':>5} {'MAE':>10} {'Comm (MB)':>12}")
print("-" * 48)
for r in sorted(rows, key=lambda x: (x["algorithm"], x["alpha"])):
    key = (r["algorithm"], r["alpha"])
    c = costs.get(key, 0)
    print(f"{r['algorithm']:<18} {r['alpha']:>5.1f} "
          f"{r['unscaled_mae']:>10.4f} {c:>12.1f}")

## 4. Per-Deployment Performance Equity

**Key plot for the TFG**: Does FedGen reduce performance disparity 
across deployments compared to FedAvg?

In [ ]:
fig, axes = uc2.plot_per_client_equity(rows)
plt.savefig(os.path.join(uc2.RESULTS, "per_deployment_equity.pdf"),
            bbox_inches="tight", dpi=150)
plt.show()

## 5. Equity Metric: Coefficient of Variation

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

by_alg = defaultdict(list)
for r in rows:
    if r["per_user_mae_cv"] is not None:
        by_alg[r["algorithm"]].append(r)

for alg, data in by_alg.items():
    data = sorted(data, key=lambda x: x["alpha"])
    alphas = [d["alpha"] for d in data]
    cvs = [d["per_user_mae_cv"] for d in data]
    ax.plot(alphas, cvs,
            marker=uc2.MARKERS.get(alg, "o"),
            color=uc2.COLORS.get(alg, "#333"),
            label=alg, linewidth=2, markersize=8)

ax.set_xlabel("Dirichlet α (higher = more IID)")
ax.set_ylabel("CV of per-user MAE (lower = more equitable)")
ax.set_title("Performance Equity Across Deployments")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(uc2.RESULTS, "equity_cv.pdf"),
            bbox_inches="tight", dpi=150)
plt.show()

## 6. Training Convergence Comparison

In [ ]:
# One subplot per alpha
fig, axes = plt.subplots(1, len(uc2.ALPHA_SWEEP), 
                         figsize=(5*len(uc2.ALPHA_SWEEP), 5), sharey=True)

for ax, alpha in zip(axes, uc2.ALPHA_SWEEP):
    for alg in ["Centralized", "FedAvg", "FedGen", "FedGen_partial"]:
        key = (alg, alpha)
        if key not in results:
            continue
        r = results[key]
        glob_metrics = r["metrics"].get("glob_test_metric", [])
        maes = [m.get("unscaled_mae") for m in glob_metrics]
        ax.plot(maes, color=uc2.COLORS.get(alg, "#333"),
                label=alg, linewidth=1.5, alpha=0.8)
    ax.set_title(f"α = {alpha}")
    ax.set_xlabel("Round")
    if ax == axes[0]:
        ax.set_ylabel("Unscaled MAE (MB)")
    ax.legend(fontsize=8)

plt.suptitle("Training Convergence by Heterogeneity Level", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(uc2.RESULTS, "convergence_all.pdf"),
            bbox_inches="tight", dpi=150)
plt.show()

## 7. Summary Table for TFG

In [ ]:
print("\n" + "="*80)
print("SUMMARY TABLE — UC2: Wi-Fi AP Load Prediction")
print("="*80)
print(f"\n{'Algorithm':<18} {'α':>5} {'MAE':>10} {'MAPE':>10} "
      f"{'Comm(MB)':>10} {'CV':>8} {'Rounds':>8}")
print("-"*72)

for r in sorted(rows, key=lambda x: (x["algorithm"], x["alpha"])):
    key = (r["algorithm"], r["alpha"])
    c = costs.get(key, 0)
    cv = r["per_user_mae_cv"]
    cv_str = f"{cv:.3f}" if cv is not None else "N/A"
    mape = r.get("unscaled_mape")
    mape_str = f"{mape:.4f}" if mape is not None else "N/A"
    
    print(f"{r['algorithm']:<18} {r['alpha']:>5.1f} "
          f"{r['unscaled_mae']:>10.4f} {mape_str:>10} "
          f"{c:>10.1f} {cv_str:>8} {r['n_rounds']:>8}")

## 8. Cross-UC Comparison Notes (for TFG narrative)

| Aspect | UC1 (Healthcare) | UC2 (Wi-Fi) |
|--------|------------------|-------------|
| Task | Classification (readmission) | Regression (load prediction) |
| Metric | AUC | MAE / MAPE |
| Data | Tabular, mixed types | Time series, 6 features |
| Model | MLP | LSTM |
| Clients | 5 hospitals | 20 deployments |
| Key finding (expected) | FedGen improves AUC equity | FedGen improves MAE + comm |
| Pareto | FedGen dominates on AUC-comm | FedGen partial dominates on MAE-comm |

The parallel analysis across both use cases strengthens the thesis 
argument that FedGen's advantages generalize across domains.